# Workflow TruFor + ViT de refinamiento (Colab)

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU (T4)**.

Fuentes de datos (se combinan en `/content/data`):
- **IMD2020** descargado de Kaggle y convertido con `prepare_imd2020.py` (prefijo `imd_`).
- **Datos propios** en Drive (`MyDrive/pdi/data/images` y `masks`), **nombrados con prefijo `own_`** (ediciones con Gemini, contraste, saturación). Quedan como *test externo*: el modelo nunca entrena con ellos.

Lo que importa vive en Drive (`BASE`): cache de TruFor y checkpoints. El disco de Colab se borra al terminar la sesión.
Las celdas de **extracción** y **entrenamiento** son *reanudables*: si la sesión se corta, volvé a correr las celdas 1 a 3 y luego la que se interrumpió.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/<usuario>/Deepfake_Detection_PDI_2026.git"   # <- COMPLETAR
BASE = "/content/drive/MyDrive/pdi"        # datos, cache y checkpoints
VARIANTS = "raw"                            # variantes separadas por espacio, ej: "raw whites-15"

## 1. Repo, dependencias y modelo TruFor
Se omiten `torch`/`torchvision` del `requirements.txt` porque Colab ya trae PyTorch con CUDA (reinstalarlo suele romper la GPU).
Los archivos nuevos (`extract_maps.py`, `preprocess.py`, `dataset.py`, `model.py`, `train.py`) tienen que estar en `src/` del repo.

In [ ]:
%cd /content
!test -d repo || git clone {REPO_URL} repo
%cd /content/repo
!python --version
!grep -viE '^(torch|torchvision)\b' requirements.txt > /tmp/req.txt
!pip install -q -r /tmp/req.txt
!bash scripts/get_trufor.sh

## 2. Verificación del entorno
Tiene que dar aprox. `0.19 / 0.20` para las dos *pristine* y `0.996 / 0.998` para las dos *tampered* (en GPU los decimales pueden variar un poco).
Si no coincide, no sigas: algo del entorno está mal.

In [ ]:
import torch
print(torch.__version__, "| cuda:", torch.cuda.is_available())
!python src/detect.py third_party/trufor/images/

## 3. Datos
### 3a. IMD2020 desde Kaggle
Dataset: `saikotahmed/imd2020` (carpetas `img`, `mask` y `Authentic`). Si la descarga anónima falla, creá en **Secretos** de Colab (ícono de llave) `KAGGLE_USERNAME` y `KAGGLE_KEY` y activá el acceso al notebook.

In [ ]:
!pip install -q kagglehub
import os, kagglehub
try:
    imd_path = kagglehub.dataset_download("saikotahmed/imd2020")
except Exception as e:
    print("Descarga anónima falló:", e)
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    imd_path = kagglehub.dataset_download("saikotahmed/imd2020")
print(imd_path)

### 3b. Convertir al formato del proyecto
Mirá la salida: muestra ejemplos de nombres, cuántos pares img–mask encontró, tamaños que no coinciden y si las máscaras parecen invertidas (en ese caso repetí con `--invert`).

In [ ]:
!python src/prepare_imd2020.py --src {imd_path} --out /content/data --prefix imd_

### 3c. Datos propios desde Drive (opcional)
Los nombres deben empezar con `own_` (ej. `own_gemini_001.png`) para que `train.py` los separe como test externo. Se copian los bytes originales con `rsync`.

In [ ]:
!test -d {BASE}/data && rsync -a {BASE}/data/ /content/data/ || echo "No hay datos propios en Drive: se usa solo IMD2020"
!ls /content/data/images | wc -l
!ls /content/data/masks | wc -l
!ls /content/data/images | grep -c '^own_' || true

## 4. Extracción de mapas TruFor (reanudable)
Guarda `cache/<variante>/<stem>.npz` en Drive. Si se corta, volvé a ejecutar esta celda: saltea lo ya procesado.

In [ ]:
!python src/extract_maps.py --images /content/data/images --cache {BASE}/cache --variants {VARIANTS}

## 5. Entrenamiento del ViT propio, desde cero (reanudable)
En cada época imprime F1/IoU del ViT **y** de TruFor puro sobre la validación (IMD2020) y, aparte, sobre el **test externo** (`own_*`).

> ⚠️ Según el README del proyecto, TruFor se entrenó con IMD2020: sus resultados sobre ese dataset están **inflados** y no representan casos nuevos. El número honesto es el del test externo.

Checkpoints en `{BASE}/runs/vit/` (`last.pt`, `best.pt`, `config.json`). Desde cero conviene probar primero con `--epochs 3`; la comparación seria, 100 o más.
Para el modo que parte del baseline de TruFor, agregá `--residual` y usá otra carpeta en `--out`.

In [ ]:
!python src/train.py --root /content/data --cache {BASE}/cache --variants {VARIANTS} --out {BASE}/runs/vit --test-prefix own_ --epochs 100

## 6. Ejemplos: original | máscara real | TruFor | ViT  (primeras 3 filas: validación IMD2020; siguientes: test externo)

In [ ]:
import sys, json
sys.path.insert(0, "src")
import torch
import matplotlib.pyplot as plt
from dataset import list_images, split, ForensicsDataset, MEAN, STD
from model import ViTRefiner

cfg = json.load(open(f"{BASE}/runs/vit/config.json"))   # mismos hiperparametros que el entrenamiento
V, SIZE = cfg["variants"], cfg["size"]
all_p = list_images("/content/data")
test_p = [p for p in all_p if cfg.get("test_prefix") and p.stem.startswith(cfg["test_prefix"])]
_, val_p = split([p for p in all_p if p not in set(test_p)])
mkds = lambda ps: ForensicsDataset("/content/data", f"{BASE}/cache", ps, V, SIZE)
ds_val, ds_test = mkds(val_p), (mkds(test_p) if test_p else None)
dev = "cuda" if torch.cuda.is_available() else "cpu"
model = ViTRefiner(3 + 2 * len(V), SIZE, cfg["patch"], cfg["dim"], cfg["depth"], cfg["heads"],
                   residual=cfg["residual"]).to(dev)
model.load_state_dict(torch.load(f"{BASE}/runs/vit/best.pt", map_location=dev))
model.eval()

def pick(ds, n):
    out = []
    for i in range(len(ds)):
        x, y = ds[i]
        if y.sum() > 0:
            out.append((x, y))
        if len(out) == n:
            break
    return out

shown = pick(ds_val, 3) + (pick(ds_test, 3) if ds_test is not None else [])   # 3 de validacion + 3 de test externo

fig, axes = plt.subplots(len(shown), 4, figsize=(14, 3.4 * len(shown)))
for r, (x, y) in enumerate(shown):
    with torch.no_grad():
        p = torch.sigmoid(model(x[None].to(dev)))[0, 0].cpu()
    panels = [("original", (x[:3] * STD + MEAN).permute(1, 2, 0).clamp(0, 1)),
              ("máscara real", y[0]), ("TruFor", x[3]), ("ViT", p)]
    for col, (t, im) in enumerate(panels):
        axes[r, col].imshow(im, cmap=None if col == 0 else "gray", vmin=0, vmax=1)
        axes[r, col].set_title(t)
        axes[r, col].axis("off")
plt.tight_layout()
plt.show()